In [ ]:
using Random
using Statistics
using Printf
using LinearAlgebra
using Plots
using Logging

function find_project_root(start::AbstractString=pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()

if !isdefined(Main, :nb_paths)
    include(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))
end

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "harmonic_oscillator_1d", "dmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

if !isdefined(Main, :System1D)
    include(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
end
using .System1D

default(; dpi=170)
nothing


## Model and DMC Parameters

This notebook runs plain diffusion Monte Carlo for a one-dimensional harmonic oscillator with
`V(x) = 0.5 * omega^2 * x^2` and open boundaries.

Parameters used below:
- Oscillator frequency `omega = 1.0`
- Time step `dt = 5.0e-3`
- Total steps `nsteps = 500`
- Equilibration steps `nequil = 100`
- Target population `targetN = 4000`
- Branching cap `branch_cap = 10`

Trial / node structure:
- Guiding policy: `NoGuiding()`
- Node policy: `NoNode()`


## Julia Construction

The next cell defines the oscillator Hamiltonian, the DMC parameters, the initialization cloud, and the notebook toggles.

Keep the construction cell as the single place where run-time controls such as debug output or CSV export are changed.


In [ ]:
omega = 1.0
V(R) = 0.5 * omega^2 * R[1]^2
H = Hamiltonian(1, 0.5, V)

targetN = 4000
dt = 5.0e-3
nsteps = 500
nequil = 100
ET0 = 0.5 * omega
branch_cap = 10
nblocks = 50

params = DMCParams(; dt=dt, nsteps=nsteps, nequil=nequil, targetN=targetN, ET0=ET0, population_control_gain=1.0, branch_cap=branch_cap, nblocks=nblocks)

rng_init = MersenneTwister(1234)
initial_positions = [[2 * rand(rng_init) - 1] for _ in 1:targetN]

SNAPSHOT_STEPS = nb_default_snapshot_steps(nsteps)
NBINS = 140
DENSITY_SMOOTHING = 9
DENSITY_RANGE = (-4.0, 4.0)
X_AXIS_LABEL = "x"
PERIOD_MARKERS = nothing

RUN_LABEL = "unguided"
RUN_COLOR = :navy
PLOT_TITLE = "Harmonic oscillator DMC"
DENSITY_TITLE = "Harmonic oscillator DMC: snapshot densities"
PLOT_MODE = :snapshot_density

SHOW_PROGRESS = false
PROGRESS_EVERY = 0
DEBUG_MODE = false
DEBUG_EVERY = 20
WRITE_RUN_CSV = false
CSV_FILENAME = "harmonic_oscillator_dmc.csv"
SAVE_FIGURES = false
FIGURE_STEM = "harmonic_oscillator_dmc"


In [ ]:
sim = run_dmc(
    H,
    params,
    initial_positions;
    rng=MersenneTwister(42),
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABEL,
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

start_idx = min(params.nequil + 1, length(sim.energy_mean_history))
mean_energy, sem_energy = nb_mean_sem(sim.energy_mean_history[start_idx:end])

println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", RUN_LABEL, params.nequil, mean_energy, sem_energy))
println("final walker population = ", sim.population_history[end])

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    nb_write_csv(csv_path, nb_dmc_rows(RUN_LABEL, sim))
    println("Wrote run CSV to: ", abspath(csv_path))
end

SIMS = [sim]
SIM_LABELS = [RUN_LABEL]
SIM_COLORS = [RUN_COLOR]


In [ ]:
history_fig = nb_plot_dmc_history(SIMS; labels=SIM_LABELS, colors=SIM_COLORS, title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

if DENSITY_RANGE === nothing
    coord_values = Float64[]
    for sim in SIMS
        append!(coord_values, nb_all_coordinates(sim; coord=1))
    end
    xlo, xhi = nb_padded_limits(coord_values; pad_frac=0.12)
else
    xlo, xhi = DENSITY_RANGE
end

density_fig = plot(
    xlabel=X_AXIS_LABEL,
    ylabel="density",
    title=DENSITY_TITLE,
    legend=:topright,
    xlims=(xlo, xhi),
)

if PLOT_MODE == :snapshot_density
    base_sim = SIMS[1]
    available_steps = SNAPSHOT_STEPS[1:min(length(SNAPSHOT_STEPS), length(base_sim.walker_positions_history))]
    for (snapshot, step_idx) in zip(base_sim.walker_positions_history, available_steps)
        centers, density = nb_density_curve_from_snapshot(
            snapshot;
            coord=1,
            nbins=NBINS,
            xmin=xlo,
            xmax=xhi,
            smoothing_window=DENSITY_SMOOTHING,
        )
        plot!(density_fig, centers, density; label="step $(step_idx)", color=SIM_COLORS[1], linewidth=2.2, alpha=0.85)
    end
else
    for (sim, label, color) in zip(SIMS, SIM_LABELS, SIM_COLORS)
        centers, density = nb_density_curve_from_snapshot(
            nb_last_snapshot(sim);
            coord=1,
            nbins=NBINS,
            xmin=xlo,
            xmax=xhi,
            smoothing_window=DENSITY_SMOOTHING,
        )
        plot!(density_fig, centers, density; label=label, color=color, linewidth=2.4)
    end
end

if PERIOD_MARKERS !== nothing
    for (k, xmark) in enumerate(PERIOD_MARKERS)
        vline!(density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
    end
end

display(density_fig)
nb_save_figure(density_fig, PATHS.figures_dir, FIGURE_STEM, "density"; enabled=SAVE_FIGURES)
